In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from kemaodanh_oai.features import (
    FEATURE_NAMES,
    extract_pair_features,
)

print("Imports successful!")
print(f"Number of features: {len(FEATURE_NAMES)}")

Imports successful!
Number of features: 32


In [4]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "configs"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Config directory:", CONFIG_DIR)


Project root: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI
Data directory: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\data
Config directory: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\configs


In [5]:
print("Data files:")

for path in sorted(DATA_DIR.rglob("*.csv")):
    print(path.relative_to(PROJECT_ROOT))

print("\nConfig files:")

for path in sorted(CONFIG_DIR.glob("*.csv")):
    print(path.relative_to(PROJECT_ROOT))


Data files:
data\data\private_test\private_test\pairs.csv
data\data\private_test\private_test\pairs_results.csv
data\data\public_test\pairs.csv
data\data\train\pairs.csv

Config files:
configs\development_split.csv


In [6]:
TRAIN_DIR = DATA_DIR / "data" / "train"
TRAIN_CSV = TRAIN_DIR / "pairs.csv"
SPLIT_CSV = CONFIG_DIR / "development_split.csv"

train_df = pd.read_csv(
    TRAIN_CSV,
    dtype={"pair_id": str},
)

split_df = pd.read_csv(
    SPLIT_CSV,
    dtype={"pair_id": str},
)

print("Training data shape:", train_df.shape)
print("Development split shape:", split_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nDevelopment split columns:")
print(split_df.columns.tolist())

display(train_df.head())
display(split_df.head())


Training data shape: (1000, 4)
Development split shape: (1000, 3)

Training columns:
['pair_id', 'image_0', 'image_1', 'fake_position']

Development split columns:
['pair_id', 'split', 'fold']


,pair_id,image_0,image_1,fake_position
0,00001,images/00295.jpg,images/01131.jpg,1
1,00002,images/00684.jpg,images/00888.jpg,0
2,00003,images/00016.jpg,images/00059.jpg,0
3,00004,images/01000.jpg,images/00449.jpg,1
4,00005,images/01117.jpg,images/01089.jpg,1


,pair_id,split,fold
0,00720,development,1
1,00610,development,2
2,00501,development,2
3,00796,development,0
4,00576,development,0


In [7]:
sample = train_df.iloc[0]

image_0_path = TRAIN_DIR / sample["image_0"]
image_1_path = TRAIN_DIR / sample["image_1"]

print("Image 0:", image_0_path)
print("Exists:", image_0_path.exists())

print("\nImage 1:", image_1_path)
print("Exists:", image_1_path.exists())

assert image_0_path.exists(), "Image 0 not found"
assert image_1_path.exists(), "Image 1 not found"

print("\nTraining image paths verified!")


Image 0: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\data\data\train\images\00295.jpg
Exists: True

Image 1: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\data\data\train\images\01131.jpg
Exists: True

Training image paths verified!


In [8]:
df = train_df.merge(
    split_df,
    on="pair_id",
    how="left",
    validate="one_to_one",
)

assert len(df) == len(train_df)
assert df["split"].notna().all()

print("Merged shape:", df.shape)

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nFold distribution:")
print(
    df.groupby(["split", "fold"])
    .size()
)

display(df.head())


Merged shape: (1000, 6)

Split distribution:
split
development    800
holdout        200
Name: count, dtype: int64

Fold distribution:
split        fold
development   0      267
              1      267
              2      266
holdout      -1      200
dtype: int64


,pair_id,image_0,image_1,fake_position,split,fold
0,00001,images/00295.jpg,images/01131.jpg,1,development,1
1,00002,images/00684.jpg,images/00888.jpg,0,development,1
2,00003,images/00016.jpg,images/00059.jpg,0,holdout,-1
3,00004,images/01000.jpg,images/00449.jpg,1,development,1
4,00005,images/01117.jpg,images/01089.jpg,1,development,2


In [10]:
dev_df = df[
    df["split"] == "development"
].copy()

holdout_df = df[
    df["split"] == "holdout"
].copy()

dev_df["fold"] = dev_df["fold"].astype(int)

print("Development pairs:", len(dev_df))
print("Holdout pairs:", len(holdout_df))

print("\nDevelopment fold distribution:")
print(dev_df["fold"].value_counts().sort_index())

assert len(dev_df) == 800
assert len(holdout_df) == 200
assert set(dev_df["fold"].unique()) == {0, 1, 2}

print("\nDevelopment split verified!")


Development pairs: 800
Holdout pairs: 200

Development fold distribution:
fold
0    267
1    267
2    266
Name: count, dtype: int64

Development split verified!


In [11]:
print("Overall development labels:")
print(dev_df["fake_position"].value_counts().sort_index())

print("\nLabels by fold:")
display(
    pd.crosstab(
        dev_df["fold"],
        dev_df["fake_position"],
        margins=True,
    )
)


Overall development labels:
fake_position
0    400
1    400
Name: count, dtype: int64

Labels by fold:


fake_position,0,1,All
fold,,,
0,133,134,267
1,134,133,267
2,133,133,266
All,400,400,800


In [12]:
from tqdm.auto import tqdm

CACHE_DIR = PROJECT_ROOT / "outputs" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = CACHE_DIR / "development_features_32.npz"

dev_df = dev_df.sort_values("pair_id").reset_index(drop=True)

if CACHE_PATH.exists():
    print("Loading cached features...")

    cached = np.load(CACHE_PATH)

    X_dev = cached["X"]
    cached_ids = cached["pair_ids"].astype(str)

    assert np.array_equal(
        cached_ids,
        dev_df["pair_id"].to_numpy()
    ), "Cached pair IDs do not match development data"

else:
    print("Extracting 32 features per pair...")

    X_dev = np.stack([
        extract_pair_features(row, TRAIN_DIR)
        for _, row in tqdm(
            dev_df.iterrows(),
            total=len(dev_df),
            desc="Feature extraction"
        )
    ])

    np.savez_compressed(
        CACHE_PATH,
        X=X_dev,
        pair_ids=dev_df["pair_id"].to_numpy(),
    )

    print("Cache saved:", CACHE_PATH)

y_dev = dev_df["fake_position"].to_numpy(dtype=int)
folds = dev_df["fold"].to_numpy(dtype=int)

print("\nFeature matrix:", X_dev.shape)
print("Labels:", y_dev.shape)
print("Folds:", folds.shape)

assert X_dev.shape == (800, 32)
assert np.isfinite(X_dev).all()

print("\nFeature extraction completed!")


c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Extracting 32 features per pair...


Feature extraction: 100%|██████████| 800/800 [01:04<00:00, 12.35it/s]

Cache saved: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\outputs\cache\development_features_32.npz

Feature matrix: (800, 32)
Labels: (800,)
Folds: (800,)

Feature extraction completed!


In [13]:
features_df = pd.DataFrame(
    X_dev,
    columns=FEATURE_NAMES,
)

print("Feature matrix shape:", features_df.shape)

print("\nMissing values:")
print(features_df.isna().sum().sum())

print("\nFeature statistics:")
display(features_df.describe().T)

print("\nFirst 5 pairs:")
display(features_df.head())


Feature matrix shape: (800, 32)

Missing values:
0

Feature statistics:


,count,mean,std,min,25%,50%,75%,max
log_bytes,800.0,-0.004669,0.339352,-0.998226,-0.229501,0.006536,0.214294,1.061470
mean_r,800.0,0.001590,0.160745,-0.531054,-0.106873,-0.005889,0.110784,0.524528
mean_g,800.0,0.006019,0.151364,-0.419277,-0.094406,0.004987,0.110687,0.520031
mean_b,800.0,0.010681,0.155959,-0.405833,-0.095289,0.012736,0.118245,0.425604
std_r,800.0,-0.000230,0.075703,-0.239849,-0.050965,0.001321,0.051591,0.258911
std_g,800.0,0.003476,0.068620,-0.205270,-0.037945,0.003732,0.047274,0.210077
std_b,800.0,0.005532,0.076179,-0.259152,-0.042553,0.005538,0.057545,0.216130
q10_r,800.0,-0.004752,0.161912,-0.752941,-0.109804,-0.003922,0.098039,0.643137
q10_g,800.0,-0.002934,0.123282,-0.474510,-0.075490,0.000000,0.075490,0.435294
q10_b,800.0,-0.001910,0.108830,-0.345098,-0.071569,0.000000,0.070588,0.372549



First 5 pairs:


,log_bytes,mean_r,mean_g,mean_b,std_r,std_g,std_b,q10_r,q10_g,q10_b,...,lap_std,lap_abs,residual_std,residual_abs,block_x,block_y,center_mean,center_std,border_mean,border_std
0,0.344641,0.116427,0.066591,-0.055064,-0.036234,0.000300,-0.011661,0.152941,0.117647,0.035294,...,0.025781,0.016481,0.007871,0.005633,0.010351,0.008871,0.016254,0.027881,0.096492,-0.009097
1,0.064710,0.068097,0.126174,0.066602,0.074292,0.091300,0.062482,-0.105882,-0.082353,-0.047059,...,0.045215,0.014455,0.014752,0.004494,0.003345,0.005726,0.054380,0.108228,0.128595,0.075118
2,0.158539,-0.256206,-0.184247,-0.208699,-0.005162,-0.038761,-0.053251,-0.215686,-0.160784,-0.184314,...,0.031153,0.008362,0.007695,0.002970,0.004026,0.008266,-0.192107,-0.009448,-0.218103,-0.047213
3,-0.177042,-0.156646,0.017695,0.091131,0.001022,-0.039589,-0.021207,-0.145098,0.082353,0.074510,...,0.010005,0.003331,0.001066,0.000463,0.000759,-0.000325,0.035476,-0.039833,-0.060592,0.018686
4,0.553111,0.166696,0.050472,0.024163,0.108771,0.077591,0.089930,-0.090196,-0.129412,-0.149020,...,0.061209,0.025668,0.020268,0.008722,0.009614,0.014499,0.102760,0.063807,0.070762,0.077645


In [15]:
from sklearn.base import clone

base_model = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        C=0.1,
        fit_intercept=False,
        max_iter=5000,
        random_state=42,
    ),
)

oof_probs = np.full(len(dev_df), np.nan)
fold_scores = []
trained_models = {}

for fold in sorted(np.unique(folds)):
    train_mask = folds != fold
    valid_mask = folds == fold

    X_train = X_dev[train_mask]
    y_train = y_dev[train_mask]

    X_valid = X_dev[valid_mask]
    y_valid = y_dev[valid_mask]

    model = clone(base_model)
    model.fit(X_train, y_train)

    valid_probs = model.predict_proba(X_valid)[:, 1]
    valid_preds = (valid_probs >= 0.5).astype(int)

    oof_probs[valid_mask] = valid_probs
    trained_models[int(fold)] = model

    fold_f1 = f1_score(
        y_valid,
        valid_preds,
        average="macro",
    )

    fold_acc = accuracy_score(
        y_valid,
        valid_preds,
    )

    fold_scores.append({
        "fold": int(fold),
        "n_valid": len(y_valid),
        "macro_f1": fold_f1,
        "accuracy": fold_acc,
    })

    print(
        f"Fold {fold}: "
        f"Macro-F1 = {fold_f1:.4f}, "
        f"Accuracy = {fold_acc:.4f}"
    )

assert np.isfinite(oof_probs).all()

oof_preds = (oof_probs >= 0.5).astype(int)

oof_macro_f1 = f1_score(
    y_dev,
    oof_preds,
    average="macro",
)

oof_accuracy = accuracy_score(
    y_dev,
    oof_preds,
)

print("\n========== OVERALL OOF ==========")
print(f"Macro-F1: {oof_macro_f1:.4f}")
print(f"Accuracy: {oof_accuracy:.4f}")

display(pd.DataFrame(fold_scores))


Fold 0: Macro-F1 = 0.7969, Accuracy = 0.7978
Fold 1: Macro-F1 = 0.7602, Accuracy = 0.7603
Fold 2: Macro-F1 = 0.8007, Accuracy = 0.8008

========== OVERALL OOF ==========
Macro-F1: 0.7862
Accuracy: 0.7863


,fold,n_valid,macro_f1,accuracy
0,0,267,0.796930,0.797753
1,1,267,0.760216,0.760300
2,2,266,0.800681,0.800752


In [16]:
print("Classification report:")
print(
    classification_report(
        y_dev,
        oof_preds,
        target_names=["Fake at 0", "Fake at 1"],
        digits=4,
    )
)

print("\nConfusion matrix:")
cm = confusion_matrix(y_dev, oof_preds)

display(
    pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"],
    )
)

print("\nNumber of errors:")
n_errors = int((oof_preds != y_dev).sum())
print(f"{n_errors} / {len(y_dev)}")

print("\nReference Macro-F1: 0.7887")
print(f"Our Macro-F1:       {oof_macro_f1:.4f}")
print(f"Difference:         {oof_macro_f1 - 0.7887:+.4f}")


Classification report:
              precision    recall  f1-score   support

   Fake at 0     0.7746    0.8075    0.7907       400
   Fake at 1     0.7990    0.7650    0.7816       400

    accuracy                         0.7863       800
   macro avg     0.7868    0.7863    0.7862       800
weighted avg     0.7868    0.7863    0.7862       800


Confusion matrix:


,Predicted 0,Predicted 1
Actual 0,323,77
Actual 1,94,306



Number of errors:
171 / 800

Reference Macro-F1: 0.7887
Our Macro-F1:       0.7862
Difference:         -0.0025


In [17]:
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

oof_df = pd.DataFrame({
    "pair_id": dev_df["pair_id"].to_numpy(),
    "fold": folds,
    "fake_position": y_dev,
    "prob_fake_1": oof_probs,
    "prediction": oof_preds,
})

oof_df["correct"] = (
    oof_df["prediction"] == oof_df["fake_position"]
)

oof_path = RESULTS_DIR / "lr_32features_oof.csv"
oof_df.to_csv(oof_path, index=False)

print("Saved:", oof_path)
print("OOF Macro-F1:", round(oof_macro_f1, 4))
print("OOF errors:", int((~oof_df["correct"]).sum()))


Saved: c:\Users\PC\Desktop\Python\KeMaoDanh-OAI\outputs\results\lr_32features_oof.csv
OOF Macro-F1: 0.7862
OOF errors: 171
